# 4.8 Kaggle房价预测实战

一定要自己从头复现一遍。
#### 速记：
1. 有几个数据类型（特征）：id 面积等有效信息 房价（输出）
2. 删除不需要的特征： 删掉id

---
## 开始
先导库导文件。我也不太懂。后面要记得看懂。

In [6]:
import hashlib
import os
import tarfile
import zipfile
import requests

#@save
DATA_HUB = dict()
DATA_URL = 'http://d2l-data.s3-accelerate.amazonaws.com/'

def download(name, cache_dir=os.path.join('..', 'data')):  #@save
    """下载一个DATA_HUB中的文件，返回本地文件名"""
    assert name in DATA_HUB, f"{name} 不存在于 {DATA_HUB}"
    url, sha1_hash = DATA_HUB[name]
    os.makedirs(cache_dir, exist_ok=True)
    fname = os.path.join(cache_dir, url.split('/')[-1])
    if os.path.exists(fname):
        sha1 = hashlib.sha1()
        with open(fname, 'rb') as f:
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        if sha1.hexdigest() == sha1_hash:
            return fname  # 命中缓存
    print(f'正在从{url}下载{fname}...')
    r = requests.get(url, stream=True, verify=True)
    with open(fname, 'wb') as f:
        f.write(r.content)
    return fname

def download_extract(name, folder=None):  #@save
    """下载并解压zip/tar文件"""
    fname = download(name)
    base_dir = os.path.dirname(fname)
    data_dir, ext = os.path.splitext(fname)
    if ext == '.zip':
        fp = zipfile.ZipFile(fname, 'r')
    elif ext in ('.tar', '.gz'):
        fp = tarfile.open(fname, 'r')
    else:
        assert False, '只有zip/tar文件可以被解压缩'
    fp.extractall(base_dir)
    return os.path.join(base_dir, folder) if folder else data_dir

def download_all():  #@save
    """下载DATA_HUB中的所有文件"""
    for name in DATA_HUB:
        download(name)

In [7]:
%matplotlib inline
import numpy as np
import pandas as pd
import torch
from torch import nn
from d2l import torch as d2l


DATA_HUB['kaggle_house_train'] = (  #@save
    DATA_URL + 'kaggle_house_pred_train.csv',
    '585e9cc93e70b39160e7921475f9bcd7d31219ce')

DATA_HUB['kaggle_house_test'] = (  #@save
    DATA_URL + 'kaggle_house_pred_test.csv',
    'fa19780a7b011d9b009e8bff8e99922a8ee2eb90')

In [8]:
train_data = pd.read_csv(download('kaggle_house_train'))
test_data = pd.read_csv(download('kaggle_house_test'))

正在从http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_train.csv下载..\data\kaggle_house_pred_train.csv...
正在从http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_test.csv下载..\data\kaggle_house_pred_test.csv...


好了。得先瞅瞅训练集和测试集长什么样子，这俩可能不太一样，得把特征矩阵形状搞一致。

In [16]:
print(train_data.shape)
print(test_data.shape)
print(train_data.iloc[0:4, [0, 1, 2, 3, -3, -2, -1]]) 
#iloc是基于位置的索引，iloc[0:4, [0, 1, 2, 3, -3, -2, -1]]表示选取前4行和指定的列（第0、1、2、3列以及倒数第3、2、1列）。
print(test_data.iloc[0:4, [0, 1, 2, 3, -3, -2, -1]])
# 训练数据的最后一列是标签（房价），而测试数据没有标签。

# 让我康康

(1460, 81)
(1459, 80)
   Id  MSSubClass MSZoning  LotFrontage SaleType SaleCondition  SalePrice
0   1          60       RL         65.0       WD        Normal     208500
1   2          20       RL         80.0       WD        Normal     181500
2   3          60       RL         68.0       WD        Normal     223500
3   4          70       RL         60.0       WD       Abnorml     140000
     Id  MSSubClass MSZoning  LotFrontage  YrSold SaleType SaleCondition
0  1461          20       RH         80.0    2010       WD        Normal
1  1462          20       RL         81.0    2010       WD        Normal
2  1463          60       RL         74.0    2010       WD        Normal
3  1464          60       RL         78.0    2010       WD        Normal


显而易见，这个id没有屁用，删掉。
> 但我有点好奇如果放在这里不动，会不会对预测造成影响捏？造成影响的话就就感觉太荒谬了。

In [ ]:
all_features = pd.concat((train_data.iloc[:, 1:-1], test_data.iloc[:, 1:]))
#concat函数用于连接两个DataFrame对象，这里将训练数据和测试数据的特征部分（去掉第一列ID和最后一列标签）进行连接，得到一个包含所有特征的DataFrame对象all_features。

在主要部分开始之前先理一下要干啥：
- 1. 洗数据  *数据预处理*
- 2. 训练（加一些技巧）
- 3. K折交叉验证
- 4. 根据验证结果调参，并调整第2步的训练过程。
---
1.  **数据预处理**

    首先统一尺度。进行**标准化**和**离散值处理**：
    - 使各类数值型列的期望相等。
    - 离散型转化为数值型。
  <math xmlns="http://www.w3.org/1998/Math/MathML" display="block">
  <mi>x</mi>
  <mo stretchy="false">&#x2190;</mo>
  <mfrac>
    <mrow>
      <mi>x</mi>
      <mo>&#x2212;</mo>
      <mi>&#x3BC;</mi>
    </mrow>
    <mi>&#x3C3;</mi>
  </mfrac>
  <mo></mo>
</math>

标准化之后再转化为张量形式。以便训练时计算。

In [ ]:
numeric_features = all_features.dtypes[all_features.dtypes != 'object'].index
#返回特征矩阵里，数值（非数值型-'object'）类型列的列名
all_features[numeric_features] = all_features[numeric_features].apply(
    lambda x: (x - x.mean()) / (x.std()))
all_features[numeric_features] = all_features[numeric_features].fillna(0)
# 在标准化数据之后，将缺失值设置为均值0
all_features = pd.get_dummies(all_features, dummy_na=True)
# 对离散特征进行独热编码，dummy_na=True表示将缺失值也作为一个类别进行编码
all_features.shape
#让我康康形状

# 将数据转换为PyTorch张量
train_features = torch.tensor(all_features[:train_data.shape[0]].values, dtype=torch.float32)
test_features = torch.tensor(all_features[train_data.shape[0]:].values, dtype=torch.float32)
train_labels = torch.tensor(train_data.SalePrice.values, dtype=torch.float32)


---
2. **训练**

    话不多说。
    - 先搞个一层线性做个参照，以防捣鼓出来开倒车
    - 损失计算
    - 训练，开导